The purpose of this notebook is to provide a tool that allows the user to download CODEX data files from a database. 

The script queries the database based on user input. By specifying a range of dates, the user can download either L1 science data files (if the keyword "calibration" is None) or L1 calibration data files (if the keyword "calibration" is not None). 

Last updated May-22-2026 by Niharika Godbole

In [ ]:
import csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import os
import re
from pathlib import Path

# URL for CODEX database.
url_L1 = "https://umbra.nascom.nasa.gov/codex/"


def is_dir_allowed(dir_url, start_date, end_date, calib_type):
    """
    Checks directories from which to download data, based on user input.
    
    Example usage: 

    START_DATE=None, END_DATE=None, CALIB_TYPE=None: 
        Downloads the entire database (all .fits files + all calibration files).

    START_DATE='2024-11-10', END_DATE='2024-11-15', CALIB_TYPE=None: 
        Downloads only the standard .fits files in the date range, skipping the calibration data. 

    START_DATE='2024-11-10', END_DATE='2024-11-15', CALIB_TYPE='dark': 
        Downloads only the 'dark' calibration data in that date range, skipping the standard .fits files.

    Input: 
        dir_url: Path object
            Path for the directory of interest. 

        start_date: string

        end_date: string

        calib_type: string
            Specifies the calibration dataset to download. 
            Valid entries include: 'dark', 'earthshine', 'star', 'window'.
            If calibration data is not desired, specify 'None'. 

    Output: Boolean mask
    """

    if start_date is None and end_date is None and calib_type is None:
        return True
        
    url_lower = dir_url.lower()
    
    if calib_type:
        # If calib_type is specified, ignore the top-level standard L1 year folders (e.g. /codex/2024/)
        if re.search(r'/codex/\d{4}/', url_lower):
            return False
        # If we are inside /calibration/, only enter the specified calibration folder
        match = re.search(r'/calibration/([^/]+)/', url_lower)
        if match:
            current_calib = match.group(1)
            if current_calib != calib_type.lower():
                return False
    else:
        # If no calib_type is specified (but dates ARE specified), ignore the calibration folder entirely
        if '/calibration/' in url_lower:
            return False
                
    if start_date and end_date:
        start_y, start_m = start_date.split('-')[0], start_date.split('-')[1]
        end_y, end_m = end_date.split('-')[0], end_date.split('-')[1]
        
        # Check Year directory (e.g. /2024/)
        year_match = re.search(r'/(\d{4})/$', url_lower)
        if year_match:
            y = year_match.group(1)
            if y < start_y or y > end_y:
                return False
                
        # Check Month directory (e.g. /2024/11/)
        month_match = re.search(r'/(\d{4})/(\d{2})/$', url_lower)
        if month_match:
            ym = f"{month_match.group(1)}-{month_match.group(2)}"
            start_ym, end_ym = f"{start_y}-{start_m}", f"{end_y}-{end_m}"
            if ym < start_ym or ym > end_ym:
                return False
                
        # Check Day directory (e.g. /2024/11/13/)
        day_match = re.search(r'/(\d{4})/(\d{2})/(\d{2})/$', url_lower)
        if day_match:
            ymd = f"{day_match.group(1)}-{day_match.group(2)}-{day_match.group(3)}"
            if ymd < start_date or ymd > end_date:
                return False
                
    return True


def is_file_allowed(file_url, start_date, end_date, calib_type):
    """
    Checks files to download based on user input. 

    Example usage: 

    START_DATE=None, END_DATE=None, CALIB_TYPE=None: 
        Downloads the entire database (all .fits files + all calibration files).

    START_DATE='2024-11-10', END_DATE='2024-11-15', CALIB_TYPE=None: 
        Downloads only the standard .fits files in the date range, skipping the calibration data. 

    START_DATE='2024-11-10', END_DATE='2024-11-15', CALIB_TYPE='dark': 
        Downloads only the 'dark' calibration data in that date range, skipping the standard .fits files.

    Input: 
        file_url: Path object
            Path for the directory of interest. 

        start_date: string

        end_date: string

        calib_type: string
            Specifies the calibration dataset to download. 
            Valid entries include: 'dark', 'earthshine', 'star', 'window'.
            If calibration data is not desired, specify 'None'. 

    Output: Boolean mask
    """

    if start_date is None and end_date is None and calib_type is None:
        return True
        
    url_lower = file_url.lower()
    
    if calib_type:
        if f"/calibration/{calib_type.lower()}/" not in url_lower:
            return False
    else:
        # Ensure no calibration files slip through if calib_type is None (but dates ARE specified)
        if '/calibration/' in url_lower:
            return False
            
    if start_date and end_date:
        # Extract date from URL (looks like /2024/11/13/)
        match = re.search(r'/(\d{4})/(\d{2})/(\d{2})/', url_lower)
        if match:
            file_date_str = f"{match.group(1)}-{match.group(2)}-{match.group(3)}"
            if not (start_date <= file_date_str <= end_date):
                return False
                
    return True


def scrape_directory(url, writer, start_date=None, end_date=None, calib_type=None, visited=None, file_extension=None):
    
    if visited is None:
        visited = set()
    
    if url in visited:
        return
    visited.add(url)
    
    try:
        time.sleep(0.5) # Be respectful to server.
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        rows = soup.find_all('tr')
        
        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 3:
                link_tag = cols[1].find('a')
                if not link_tag:
                    continue
                
                name = link_tag.text.strip()
                href = link_tag.get('href', '')
                
                if name == 'Parent Directory' or name == '../':
                    continue
                
                full_url = urljoin(url, href)
                last_modified = cols[2].text.strip() if len(cols) > 2 else ''
                size = cols[3].text.strip() if len(cols) > 3 else ''
                
                if name.endswith('/'):
                    # Check if we should traverse this directory based on filters.
                    if is_dir_allowed(full_url, start_date, end_date, calib_type):
                        print(f"Entering directory: {full_url}")
                        scrape_directory(full_url, writer, start_date, end_date, calib_type, visited, file_extension)
                else:
                    if file_extension and not name.lower().endswith(file_extension.lower()):
                        continue
                    
                    # Final check before logging file.
                    if is_file_allowed(full_url, start_date, end_date, calib_type):
                        print(f"Found file: {name} | Date: {last_modified} | Size: {size}")
                        writer.writerow([full_url, name, last_modified, size])
    
    except requests.exceptions.RequestException as e:
        print(f"Error accessing {url}: {e}")
    except Exception as e:
        print(f"Error processing {url}: {e}")


def download_all_files(url, output_dir, start_date=None, end_date=None, calib_type=None, file_extension='.fits', visited=None):
    """
    Download all of the files as specified by the helper functions. 
    """
    
    if visited is None:
        visited = set()
    
    if url in visited:
        return
    visited.add(url)
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        for link in soup.find_all('a'):
            href = link.get('href', '')
            name = link.text.strip()
            
            if name == 'Parent Directory' or name == '../':
                continue
            
            full_url = urljoin(url, href)
            
            if name.endswith('/'):
                # Check if we should traverse this directory based on filters.
                if is_dir_allowed(full_url, start_date, end_date, calib_type):
                    print(f"Entering directory: {name}")
                    folder_name = name.rstrip('/')
                    new_output_dir = os.path.join(output_dir, folder_name)
                    
                    download_all_files(
                        full_url, new_output_dir, start_date, end_date, calib_type, file_extension, visited
                    )
            else:
                if not name.lower().endswith(file_extension.lower()):
                    continue
                
                # Final check before downloading.
                if not is_file_allowed(full_url, start_date, end_date, calib_type):
                    continue
                
                output_path = os.path.join(output_dir, name)
                if os.path.exists(output_path):
                    print(f"Skipping {name} (already exists)")
                    continue
                
                print(f"Downloading {name} to {output_dir}...", end=' ')
                try:
                    file_response = requests.get(full_url, timeout=60)
                    file_response.raise_for_status()
                    with open(output_path, 'wb') as f:
                        f.write(file_response.content)
                    print("✓")
                    time.sleep(0.5) 
                except Exception as e:
                    print(f"✗ Failed: {e}")
    
    except Exception as e:
        print(f"Error accessing {url}: {e}")


def main_scrape(output_dir, start_date, end_date, calib_type):
    """
    Writes files within the specified range to a .csv file for reference. 

    Sets up the scraper function. 
    """

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    output_file = os.path.join(output_dir, "codex_l1_files.csv")
    
    with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Full Path', 'Filename', 'Date Added', 'File Size'])
        print(f"\n--- Starting to scrape directory: {url_L1} ---")
        scrape_directory(url_L1, writer, start_date, end_date, calib_type, file_extension='.fits')
    print(f"Done! File information saved to {output_file}\n")

# Configure and Run the Scraper

In [ ]:
if __name__ == "__main__":
    
    # Format: 'YYYY-MM-DD'. 
    # Leave as None to download all dates.
    START_DATE = '2024-11-10' 
    END_DATE = '2024-11-15'
    
    # Options: 'dark', 'earthshine', 'star', 'window'. 
    # Leave as None for standard L1 & all calib data.
    CALIB_TYPE = 'dark'

    # Change this directory to the one that you wish to use.
    output_dir = Path(r"C:\Users\ngodbole\OneDrive - NASA\Desktop\analysis\data\codex\L1")

    # Scrape Directories to .csv file. This provides an inventory of all the files that are downloaded. 
    # This step can take a long time to complete, especially if there are many files. 
    # If you do not need this inventory, then I suggest commenting this line out. 
    main_scrape(output_dir, START_DATE, END_DATE, CALIB_TYPE)

    # Download the specified .fits files.
    print(f"Downloading .fits files from {url_L1}")
    print(f"Saving to structure inside: {output_dir}\n")
    download_all_files(url_L1, output_dir, START_DATE, END_DATE, CALIB_TYPE, file_extension='.fits')

    print("\nAll done!")